# Aula 2: tipos de variáveis

Uma pesquisa empírica em Direito termina numa tabela. Quem decide o que vira
coluna dessa tabela é você, e cada coluna é uma **variável**. Esta aula é sobre
dizer que tipo é cada variável, e sobre fazer o pandas concordar com você.

Como o notebook funciona: cada operação nova aparece primeiro resolvida, e logo
depois vem um bloco **Agora você**, que pede a mesma operação em outra coluna.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


## A pergunta de pesquisa

> Entre 2023 e 2025, como se distribuem no TJSP os processos com assunto de
> saúde (medicamento, tratamento médico-hospitalar, plano de saúde): em que grau
> tramitam, de que classe são, e quanto tempo passa entre o ajuizamento e a
> última movimentação registrada?

Repare que a pergunta já obriga a decidir coisas. "Quanto tempo" pede uma
variável numérica que **não existe** na base e vai precisar ser construída.


### De onde vieram os dados

A base saiu da API pública do DataJud (CNJ), pelo juscraper. O código abaixo é o
que foi rodado uma vez, e está aqui só para você saber a origem da tabela. Não
precisa rodar.

```python
import juscraper as jus

datajud = jus.scraper("datajud")
processos = datajud.listar_processos(
    tribunal="TJSP",
    assuntos=[6064, 6233, 7775, 10064, 10356, 12484, 12487, 12489, 14760],
    ano_ajuizamento=2024,
    paginas=1,
)
```


In [ ]:
saude = pd.read_csv(f"{URL}/tjsp_datajud_saude.csv")
saude.head()


In [ ]:
saude.shape


## Primeiro olhar: o que o pandas achou de cada coluna

`.info()` mostra, para cada coluna, quantos valores não são nulos e qual o
**dtype**, que é o tipo de armazenamento do pandas. Cuidado com a palavra
"tipo": o dtype é uma decisão que o pandas tomou ao ler o arquivo, e não a
natureza da variável.


In [ ]:
saude.info()


`.nunique()` conta quantos valores distintos existem em cada coluna. É a segunda
coisa a olhar, e já adianta muito sobre o tipo de cada variável.


In [ ]:
saude.nunique()


### O dtype não é o tipo da variável

Estas quatro colunas vieram todas como número:


In [ ]:
saude[["classe_codigo", "municipio_ibge", "nivel_sigilo", "n_assuntos"]].dtypes


E o pandas deixa você fazer esta conta sem reclamar:


In [ ]:
saude["municipio_ibge"].mean()


Saiu um número, e ele não significa nada: `municipio_ibge` é um **código**, um
rótulo que por acaso é escrito com dígitos. A média de um rótulo é um número sem
referente no mundo. O mesmo vale para `classe_codigo`.

Já `n_assuntos` é uma contagem de verdade, e a média dela responde a alguma
coisa: quantos assuntos, em média, um processo tem. Mesma aparência no arquivo,
natureza diferente.

> **Regra prática:** se somar dois valores da variável não produz nada com
> sentido, ela não é numérica, por mais que seja escrita com dígitos.


### Exercício 1: classifique as variáveis

Os tipos são estes, com um exemplo desta base para cada um:

| tipo | é assim quando | nesta base |
|---|---|---|
| `identificador` | serve para achar o caso, não para medir | `numero_processo` |
| `categorica_nominal` | rótulos sem ordem entre si | `sistema` (SAJ, Projudi) |
| `categorica_ordinal` | rótulos com ordem natural | `grau` (G1 antes de G2) |
| `categorica_binaria` | só dois valores possíveis | `formato` (eletrônico, físico) |
| `numerica_discreta` | resultado de contar | `n_assuntos` |
| `numerica_continua` | resultado de medir numa escala | nenhuma ainda, vamos criar |
| `data` | matéria-prima para criar outras | `data_ajuizamento` |

Três já estão preenchidas como modelo. Complete o resto.


In [ ]:
tipos = {
    # os três primeiros são o modelo
    "numero_processo": "identificador",
    "sistema": "categorica_nominal",
    "n_assuntos": "numerica_discreta",
    # complete daqui para baixo
    "tribunal": "________",
    "grau": "________",
    "classe": "________",
    "classe_codigo": "________",
    "assunto": "________",
    "orgao_julgador": "________",
    "municipio_ibge": "________",
    "formato": "________",
    "nivel_sigilo": "________",
    "data_ajuizamento": "data",
    "data_ultima_atualizacao": "data",
}

pd.Series(tipos).value_counts()


### Variável que não varia

Antes de qualquer conta, veja quantos valores distintos cada coluna tem.
Variável com um valor só não explica nada: ela é constante no recorte.


In [ ]:
saude[["tribunal", "nivel_sigilo", "formato"]].nunique()


In [ ]:
saude["formato"].value_counts()


`formato` é binária no papel, mas tem 3998 eletrônicos e 2 físicos.
Tecnicamente varia; na prática, não dá para comparar nada com dois casos de um
lado. Isso é decisão de pesquisa, não de programação.


## Converter: de número para texto

`.astype("string")` transforma a coluna em texto. Fazemos isso com códigos, para
que ninguém calcule média deles por acidente. Veja com `classe_codigo`:


In [ ]:
saude["classe_codigo"] = saude["classe_codigo"].astype("string")

saude["classe_codigo"].dtype


Com `municipio_ibge` tem um passo a mais. Ela veio como `float64` (repare no
`.0` no fim de cada número), porque o pandas usa `float` para poder representar
o valor faltante. Converter direto para texto guardaria o `.0` junto:


In [ ]:
saude["municipio_ibge"].head(3)


In [ ]:
saude["municipio_ibge"].astype("string").head(3)


A solução é passar antes por `"Int64"`, com I maiúsculo, que é o inteiro do
pandas que aceita valor faltante. Depois, sim, vira texto:


In [ ]:
saude["municipio_ibge"] = saude["municipio_ibge"].astype("Int64").astype("string")

saude["municipio_ibge"].head(3)


## Converter: de texto para data

No arquivo, data é texto. Enquanto for texto, `2024-03-15` é só uma sequência de
caracteres: não dá para subtrair, nem ordenar direito, nem pedir o ano.
`pd.to_datetime` faz a conversão.


In [ ]:
saude["data_ajuizamento"].head(3)


In [ ]:
saude["data_ajuizamento"] = pd.to_datetime(saude["data_ajuizamento"])

saude["data_ajuizamento"].head(3)


Repare no que mudou: o dtype passou de `object` para `datetime64[ns]`.


**Agora você.** Converta `data_ultima_atualizacao` do mesmo jeito.


In [ ]:
saude["data_ultima_atualizacao"] = pd.________(saude["data_ultima_atualizacao"])

saude[["data_ajuizamento", "data_ultima_atualizacao"]].dtypes


### O que se calcula a partir de uma data

Data não é bem um tipo da nossa lista: ela é matéria-prima. O que entra na
análise é o que se calcula a partir dela.

O acessador `.dt` dá acesso aos pedaços da data. `.dt.year` devolve o ano:


In [ ]:
saude["ano_ajuizamento"] = saude["data_ajuizamento"].dt.year

saude[["data_ajuizamento", "ano_ajuizamento"]].head(3)


**Agora você.** Crie `mes_ajuizamento` com `.dt.month`.


In [ ]:
saude["mes_ajuizamento"] = saude["data_ajuizamento"].dt.________

saude[["data_ajuizamento", "ano_ajuizamento", "mes_ajuizamento"]].head(3)


`ano_ajuizamento` sai como número inteiro, e é assim que ele fica. O que muda de
uma análise para outra não é o tipo da coluna, e sim o uso: às vezes ele entra
como número (diferença entre dois anos), às vezes como agrupador (comparar 2023,
2024 e 2025 entre si). Vale escrever qual dos dois usos você está fazendo.


### Subtraindo duas datas

Subtrair duas datas devolve um `Timedelta`, que é uma duração. Para virar número
é preciso escolher a unidade, e `.dt.days` devolve a duração em dias.


In [ ]:
(saude["data_ultima_atualizacao"] - saude["data_ajuizamento"]).head(3)


In [ ]:
saude["dias_ate_atualizacao"] = (
    saude["data_ultima_atualizacao"] - saude["data_ajuizamento"]
).dt.days

saude[["data_ajuizamento", "data_ultima_atualizacao", "dias_ate_atualizacao"]].head(3)


Pronto: a primeira variável numérica da base, e ela não existia no arquivo.
`.describe()` dá um resumo rápido de uma coluna numérica.


In [ ]:
saude["dias_ate_atualizacao"].describe()


## `pd.Categorical`: o tipo que dá trabalho

Categórica é o tipo mais comum no Direito e o mais chato de representar. Por
padrão o pandas guarda texto (`object`), o que funciona, mas perde duas coisas:
**quais são as categorias possíveis** e **se existe ordem entre elas**.

`pd.Categorical` resolve isso. Veja com `assunto`:


In [ ]:
saude["assunto"] = pd.Categorical(saude["assunto"])

saude["assunto"].dtype


A coluna agora carrega a lista de categorias, e o pandas passa a saber o
universo de valores que aquela variável pode assumir:


In [ ]:
saude["assunto"].cat.categories


**Agora você.** Faça o mesmo com a coluna `classe` e veja quantas categorias ela tem.


In [ ]:
saude["classe"] = pd.________(saude["classe"])

saude["classe"].cat.________


### Categórica ordinal

Quando existe ordem, ela precisa ser declarada, com `categories=` na ordem certa
e `ordered=True`. `grau` tem G1 (primeiro grau) e G2 (segundo grau), nessa
ordem:


In [ ]:
saude["grau_ordenado"] = pd.Categorical(
    saude["grau"], categories=["G1", "G2"], ordered=True
)

saude["grau_ordenado"].dtype


Agora repare no efeito colateral. A base tem um terceiro valor, `JE` (juizado
especial), que ficou de fora da lista de categorias:


In [ ]:
saude["grau"].unique()


In [ ]:
saude["grau_ordenado"].isna().sum()


> **Armadilha 1.** Todo valor fora da lista de categorias vira valor faltante,
> **em silêncio**. Isso é útil (declara o universo esperado) e perigoso (perde
> dado sem avisar). Confira sempre quantos viraram faltante depois de criar a
> categórica.

Aqui a perda é intencional: `JE` não fica nem antes nem depois de G1 e G2, é
outra coisa. Se quisermos manter o JE, o caminho é tratá-lo como categoria sem
ordem.


### O que a ordem permite fazer

Com `ordered=True`, comparação e ordenação passam a funcionar. Dá para comparar
a coluna com o nome de uma categoria, usando `>`, `>=`, `<` e `<=`:


In [ ]:
(saude["grau_ordenado"] >= "G2").sum()


In [ ]:
saude.sort_values("grau_ordenado")[["numero_processo", "grau"]].head(3)


Sem `ordered=True`, a mesma comparação levanta erro. Faça o teste:


In [ ]:
sem_ordem = pd.Categorical(saude["grau"], categories=["G1", "G2", "JE"])

try:
    sem_ordem >= "G2"
except TypeError as erro:
    print("TypeError:", erro)


### Valor faltante e a categoria "Outros"

A coluna `assunto` tem dois processos sem assunto informado. Juntar esses casos
numa categoria explícita costuma ser melhor do que deixá-los faltantes: a
categoria aparece nas contagens e ninguém esquece que ela existe.


In [ ]:
saude["assunto"].isna().sum()


A tentação é chamar `fillna("Outros")` direto. Não funciona, e o erro diz
exatamente por quê: `Outros` não está na lista de categorias declaradas.


In [ ]:
try:
    saude["assunto"].fillna("Outros")
except TypeError as erro:
    print("TypeError:", erro)


O caminho é abrir espaço para a categoria antes, com `add_categories`, e só
depois preencher:


In [ ]:
saude["assunto"] = saude["assunto"].cat.add_categories(["Outros"]).fillna("Outros")

saude["assunto"].isna().sum()


In [ ]:
saude["assunto"].value_counts()


> **Armadilha 2.** `value_counts()` em categórica mostra **todas** as categorias
> declaradas, inclusive as com zero ocorrências. Isso é ótimo para tabela (a
> linha existe mesmo com zero) e péssimo se você não esperava. Repare na última
> linha do resultado abaixo:


In [ ]:
segundo_grau = saude[saude["grau"] == "G2"]

segundo_grau["assunto"].value_counts()


## `pd.cut`: de numérica para categórica ordinal

Transformar uma numérica em faixas é a conversão mais comum na outra direção.
`bins` são os pontos de corte e `labels` são os nomes das faixas. O resultado já
sai como categórica **ordenada**:


In [ ]:
saude["faixa_dias"] = pd.cut(
    saude["dias_ate_atualizacao"],
    bins=[-float("inf"), 30, 180, 365, float("inf")],
    labels=["até 1 mês", "1 a 6 meses", "6 a 12 meses", "mais de 1 ano"],
)

saude["faixa_dias"].dtype


In [ ]:
saude["faixa_dias"].value_counts(sort=False)


`sort=False` mantém a ordem das faixas. Sem ele, o pandas ordenaria pela
contagem, e a tabela perderia a sequência que interessa.

Cortar em faixas **perde informação**: 31 dias e 179 dias viram a mesma coisa.
Faça isso quando a faixa for o que interessa para a pergunta, não por hábito.


**Agora você.** Como `faixa_dias` é ordenada, dá para comparar com o nome de uma faixa, do mesmo jeito que fizemos com `grau_ordenado >= "G2"`. Conte quantos processos demoraram mais de 6 meses.


In [ ]:
(saude["faixa_dias"] ________ "1 a 6 meses").sum()


## Respondendo à pergunta

Com os tipos arrumados, as contas ficam curtas. Quantos processos por assunto:


In [ ]:
saude["assunto"].value_counts()


E o tempo até a última movimentação, comparando primeiro e segundo grau. Por
enquanto fazemos isso separando a base em dois pedaços; na aula 4 você vai ver o
`groupby`, que faz esse tipo de comparação em uma linha só.


In [ ]:
primeiro_grau = saude[saude["grau"] == "G1"]

primeiro_grau["dias_ate_atualizacao"].median()


**Agora você.** Faça o mesmo para o segundo grau, usando a variável `segundo_grau` que já criamos lá em cima.


In [ ]:
segundo_grau["dias_ate_atualizacao"].________()


### Exercício 2

A coluna `orgao_julgador` tem mais de mil valores distintos. Ela é categórica
nominal, mas com tantas categorias não serve para comparar grupos. Escreva, em
duas ou três linhas, que variável derivada dela você criaria para que ela virasse
útil, e por quê. Não precisa programar.


In [ ]:
# exercício 2 (responda em célula de texto)


### Exercício 3

Crie uma faixa nova, `faixa_assuntos`, que separe os processos com um assunto dos
processos com mais de um, usando `pd.cut` sobre `n_assuntos`. Depois conte
quantos caem em cada faixa.


In [ ]:
saude["faixa_assuntos"] = pd.________(
    saude["n_assuntos"],
    bins=[0, 1, float("inf")],
    labels=["um assunto", "mais de um"],
)

saude["faixa_assuntos"].value_counts(sort=________)


## O que ficou

1. **dtype não é tipo de variável.** O pandas chuta pelo formato do arquivo; a
   natureza da variável é decisão sua, e ela decide que conta é possível.
2. **Converter é rotina.** `.astype("string")` para códigos, `"Int64"` no meio
   quando há faltantes, `pd.to_datetime` para datas.
3. **Data é matéria-prima.** O que entra na análise é o que se calcula dela, com
   `.dt.year`, `.dt.month` e subtração seguida de `.dt.days`.
4. **Categórica precisa ser declarada.** `pd.Categorical` guarda o universo de
   categorias e, com `ordered=True`, a ordem. O que fica fora vira faltante em
   silêncio, e faltante vira "Outros" com `add_categories` antes do `fillna`.
5. **`pd.cut`** faz o caminho de volta, de numérica para categórica ordenada.

Na aula 3 vamos usar esses tipos para escolher a estatística certa.
